# Exploratory Data Analysis



1. Shapes of Distributions
2. Basic Statistical Measures
3. Visualization
4. Correlation Analysis





## Shapes of Distributions

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
import seaborn as sb

# Set the ggplot style
plt.style.use("ggplot")

In [ ]:
# For later use we define a plot_dist function which can display a distribution as well as the mean and median values.
def plot_dist(
    data,
    bins=30,
    add=["mean"],
    lower_bound=None, upper_bound=None,
    figsize=(4, 4)
):
    """Enhanced plotting function to display distributions with statistics annotations,
    with options to restrict the histogram's bounds.

    Parameters:
        data (array-like): The full dataset used for statistical computations.
        bins (int): Number of histogram bins.
        add (list): List of statistics to annotate. Options: "mean", "median".
        lower_bound (float, optional): Lower bound for the histogram plot.
        upper_bound (float, optional): Upper bound for the histogram plot.
        figsize (tuple): The figure size.
    """
    def add_stat(value, height, name, color="crimson"):
        """Draw a vertical line at the statistic's value"""
        ax.axvline(value, color=color, linestyle='--', linewidth=2)

        # Check if the text is going out of bounds on the right
        if value > ax.get_xlim()[1] * 0.95:
            value = ax.get_xlim()[1] * 0.95
        ax.text(value, height, f" {name}: {value:.2f}",
                color=color,
                verticalalignment='center')

    # For histogram, only use data within the provided bounds.
    data_hist = data
    if lower_bound is not None:
        data_hist = data_hist[data_hist >= lower_bound]
    if upper_bound is not None:
        data_hist = data_hist[data_hist <= upper_bound]

    # Create the histogram using the bounded data.
    fig, ax = plt.subplots(figsize=figsize, dpi=100)
    counts, bins, patches = ax.hist(data_hist, bins=bins, rwidth=0.8,
                                    color="steelblue", alpha=0.75)

    max_height = np.max(counts)

    # Compute statistics on the full data.
    if "mean" in add:
        mean_value = np.mean(data)
        add_stat(mean_value, 0.9 * max_height, "mean")
    if "median" in add:
        median_value = np.median(data)
        add_stat(median_value, 0.8 * max_height, "median", "purple")

    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.set_title('Distribution with Statistical Annotations')
    #plt.show()

**Symmetric distributions** are those where values are distributed in a way that the shape on one side of the centerline mirrors the shape on the other. In other words, the left half of the distribution is a mirror image of the right half. One of the key properties of a symmetric distribution is that the mean and median will be the same, or very close.

In [ ]:
np.random.seed(0)
data = np.random.gamma(500, 1, 5000)
plot_dist(data, 30, ["mean", "median"])

In reality, most of the data we encounter tends to be **non-symmetric**. These distributions don’t exhibit mirror-like symmetry around their center. It is important to note here, especially for non-symmetric distributions, the mean might not necessarily represent the “center” of the data.

In [ ]:
data = np.random.gamma(1.6, 26000, 1000)
plot_dist(data, 30, ["mean", "median"])

**Outliers** are values that stand apart from the bulk of the data. Their presence can distort our perceptions about the data and can notably skew our mean. It’s essential to identify and manage outliers for better statistical interpretations.

In [ ]:
np.random.seed(0)
data = np.random.gamma(500, 1, 5000)
data[:500] = 0
plot_dist(data, 50, ["mean", "median"])

At times, our dataset may not belong to a single type of distribution. Instead, it may be the result of a mix of two or more underlying distributions. This phenomenon is observed in **mixed** distributions. Recognizing and understanding the different underlying distributions can be crucial for analysis.

In [ ]:
np.random.seed(0)
data1 = np.random.gamma(50, 2, 2000)
data2 = np.random.gamma(100, 2, 1000)
data = np.concatenate((data1, data2)) - 134

plot_dist(data, 30, ["mean", "median"])

# Statistical Measures
After observing the distributions, let’s also calculate key statistical measures and see what insights they provide regarding the distribution’s characteristics.

In [ ]:
print(np.std(data))
print(np.min(data), np.max(data))
print(np.quantile(data, 0.5))
print(np.quantile(data, 0.9))
print(np.percentile(data, [25 ,75]))

Dispersion can be illustrated with two distributions that have the same center (mean) but different standard deviations. In the following examples, both distributions are centered at 100, but they have standard deviations of 15 and 1.5, respectively. The standard deviation (often referred to as “STD”) is a measure that tells us how spread out the numbers in a distribution are.

A higher standard deviation indicates that the data points tend to be farther from the mean, while a smaller standard deviation suggests that they are clustered closely around the mean.

In [ ]:
np.random.seed(0)
data = np.random.normal(100, 15, 1000)

plot_dist(data, 30, ["mean"])
print(f"STD: {np.std(data):.2f}")
plt.xlim(52, 148)

In [ ]:
np.random.seed(1)
data = np.random.normal(100, 1.5, 1000)

plot_dist(data, 30, ["mean"])
print(f"STD: {np.std(data):.2f}")
plt.xlim(52, 148)

# Visualization

In [ ]:
import seaborn as sb

# create toy data
np.random.seed(0)

data1 = np.random.normal(4, 0.7, 150)
data2 = np.random.normal(1, 0.7, 150)
dataC = np.concatenate((data1, data2))

datasets_test = pd.DataFrame({"A": np.random.gamma(1.6, 2, 300),
                             "B": 0.6 + np.random.exponential(2.2, 300),
                             "C": dataC,
                             "D": np.random.uniform(-1, 5.5, 300)})

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 14), dpi=100)

sb.boxplot(x="variable", y="value", data=pd.melt(datasets_test), ax=ax[0][0])

sb.stripplot(x="variable", y="value", data=pd.melt(datasets_test), ax=ax[1][0], alpha=0.5, size=4)

sb.swarmplot(x="variable", y="value", data=pd.melt(datasets_test), ax=ax[0][1], size=3)

sb.violinplot(x="variable", y="value", data=pd.melt(datasets_test), ax=ax[1][1], bw_method=0.15)  # bw for "bandwidth" controls the degree of smoothing

fig.suptitle("Different ways to include distribution properties")

# Correlation Analysis

In [ ]:
# Create some toy data

# Random number generator
rng = np.random.default_rng(seed=0)

# Genereated data
a = np.arange(0, 50)
b = a + rng.integers(-10, 10, 50)
c = a + rng.integers(-20, 25, 50)
d = rng.integers(0, 50, 50)

corr_data = pd.DataFrame({"a": a,
                         "b": b,
                         "c": c,
                         "d": d})

In [ ]:
corr_data.head(3)

We created some toy data with four features (a, b, c, d). One way to look for interesting relationships between different features is simply to plot one variable (or feature) against another:

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12,3), dpi=100)

corr_data.plot(kind="scatter", x="a", y="b", ax=ax1)
corr_data.plot(kind="scatter", x="a", y="c", ax=ax2)
corr_data.plot(kind="scatter", x="a", y="d", ax=ax3)
plt.show()

Here, different variables are plotted against each other, a-b, a-c, and a-d. Most will agree that the first and second plots show a certain degree of correlation. First, between a and b, and in the second panel between a and c. The third panel shows two features that apparently are entirely unrelated in their behavior. Another thing we can see in those examples is that we should be able to measure different degrees or strengths of a correlation.

We can see the correlation in a<->b. The Pearson Correlation Coefficient measures this type of correlation very well. For this example, the values are:

In [ ]:
print(f"Corr(a, b) = {np.corrcoef(corr_data.a, corr_data.b)[1, 0]:.2f}")
print(f"Corr(a, c) = {np.corrcoef(corr_data.a, corr_data.c)[1, 0]:.2f}")
print(f"Corr(a, d) = {np.corrcoef(corr_data.a, corr_data.d)[1, 0]:.2f}")

We would thus see a strong correlation between a and b and a moderate correlation between a and c. However, there is no noticeable correlation between a and d.

The correlation coefficient allows us to quickly identify significant correlations in the data. For datasets with many variables (or features), we can easily determine the correlations for all combinations of the variables, resulting in what is called a correlation matrix.

Using Pandas we can very easily compute the Pearson correlations between all possible pairs of numerical features in a dataset:

In [ ]:
corr_data.corr()

In the following, we continue to work with generated data, but now data that is at least somewhat realistic. We want to see if body measures show any interesting correlations, and we start with (fake) data containing shoe size and height of people.

In [ ]:
filename = r"https://raw.githubusercontent.com/florian-huber/data_science_course/main/datasets/wo_men.csv"

data_people = pd.read_csv(filename)
data_people.head()

As it was introduced in the previous sections, we would often do a quick first inspection of the data using simple statistical measures.

In [ ]:
data_people.describe()

In [ ]:
# Some cleaning
mask = (data_people["shoe_size"] < 50) & (data_people["height"] > 100) & (data_people["height"] < 250)
data_people = data_people[mask]

In [ ]:
data_people.corr(numeric_only=True)

Instead of as a matrix with values, the correlation matrix is often also graphically represented, especially for larger datasets, to easily spot particularly high and low coefficients.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), dpi=100)

sb.heatmap(
    data_people.corr(numeric_only=True),
    vmin=-1, vmax=1,
    square=True, lw=2,
    annot=True, cmap="RdBu",
    ax=ax,
)
plt.show()

You can also display a scatter plot matrix to see how each pair of features are visually related.

In [ ]:
sb.pairplot(data_people, height = 2)